# 태양풍 속도 예측 — P7

P3(Public 58.8028) 를 기반으로 **장기 horizon(42h~72h) 오차**를 줄이는 데 집중한다.
앙상블·시드평균·가중치 적합은 쓰지 않는다. 단일 모델이다.

### EDA 로 확정한 사실 (`eda_p7.ipynb`)

| 항목 | 결과 | 반영 |
|---|---|---|
| 분할 구조 | validation = **109샘플 × 11블록** (9월 11개), test = Oct–Dec | 판단은 train CV 로 |
| 행 순서 | **시간 순서가 아니다** | 파일명 사슬로 시간축 복원 |
| 탄도 정렬 | 상관 최대 지연이 **12개 horizon 전부 108h** | 전제 확인. 속도를 108h 중심으로 재설정 |
| 경험적 전달속도 | **385 km/s** (P3 는 350/500/700) | `TRANSIT_SPEEDS` 교체 |
| persistence | **42h 부터 climatology 보다 나쁘다** | 42h~72h 가 표적 |
| 임계 | 0.45 가 최적 (0.3/0.6/0.75 보다 우수) | P3 값 유지 |
| 채널 | 211 ≈ AND ≫ 193 | AND 유지, 193 단독은 버림 |
| 위도 | 적도 인접 2밴드가 지배, 극지방 ≈ 0 | 위도 세분 + 적도 대칭 접기 |
| 활성영역 | bright≥1.6 이 **corr +0.128** (반대 부호) | 피처로 추가 |
| Hampel 이상치 | 윈도우 오염 **1.47%** (기준 5%) | **불필요 — 도입 안 함** |
| 원반 반지름 | 프레임별 std **6.4%** | 프레임별 robust 검출로 교체 |

### P3 대비 핵심 변경

1. **탄도 속도 재설정** — 경험적 최적 108h 를 중심으로. P3 의 v=700 은 60h 이상에서 인덱스가
   19 로 clamp 되어 **완전히 죽어 있었다**
2. **horizon 정렬 gather** — CH GRU 의 **은닉 시퀀스 전체**를 탄도 인덱스에서 보간해 head 에 넣는다.
   P3 는 horizon 별 정보가 **스칼라 3개**뿐이었다
3. **탄도 창 샘플** — 점 1개 → 소스 시각 ±1일의 4개 시점 × 중앙자오선 위도 프로파일
4. **프레임별 robust 원반 검출** — 고정 마스크가 배경을 코로나홀로 오검출하는 것을 막는다
5. **train 시간블록 CV** — validation(9월) 은 판단에 쓰지 않는다

## 1. 설정

In [ ]:
from pathlib import Path
import gc, json, math, os, random, shutil, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 777
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT_CANDIDATES = [
    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,
    Path("public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public_dataset/competition_dataset_6h"),
    Path("public/public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),
    Path("dataset"), Path("/home/jovyan/dataset"),
]
DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if candidate is not None and (candidate / "train/inputs.csv").exists():
        DATA_ROOT = candidate
        break
if DATA_ROOT is None:
    raise FileNotFoundError("데이터 경로 없음")

WORK_DIR = Path("work")
CACHE_ROOT = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "outputs_p7"
SUBMISSION_DIR = Path("submission")
for directory in (CACHE_ROOT, OUTPUT_DIR, SUBMISSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_COLUMNS = [f"image_{i:02d}" for i in range(20)]
WIND_COLUMNS = [f"wind_{i:02d}" for i in range(20)]
TARGET_COLUMNS = [f"target_{i:02d}" for i in range(12)]
HORIZONS = np.arange(1, 13) * 6
CHANNELS = ("193", "211")
AU_KM = 1.496e8

# ---- 코로나홀 추출 --------------------------------------------------------
CH_CODE_VERSION = "p7a"      # 추출 코드를 고치면 반드시 올릴 것 (캐시 오재사용 방지)
FINE_GRID = (12, 30)         # (위도, 경도) 세밀 격자. 3·5·6 으로 나눠떨어져
                             # (3,5)·(6,3)·(4,3) 등을 모두 합산으로 파생할 수 있다
CH_CUTS = (0.30, 0.45, 0.60) # 정규화값(픽셀/원반중앙값) 이하이면 코로나홀. EDA: 0.45 최적
BRIGHT_CUT = 1.60            # 이상이면 활성영역. EDA: corr +0.128
DISK_MARGIN = 0.95           # P3 값
PER_FRAME_DISK = True        # EDA: 반지름 프레임별 std 6.4% -> 프레임마다 검출

# ---- 사용할 격자 / 레벨 ---------------------------------------------------
CH_GRID = (6, 3)             # (위도, 경도). EDA+Collin: 위도 세분, 경도 축소
FOLD_LATITUDE = True         # 적도 대칭 밴드를 합친다 (Collin 의 S_ij)
USE_LEVELS = ("dark0.45", "bright")   # CH 브랜치에 넣을 레벨

# ---- 탄도 정렬 (EDA 로 재설정) --------------------------------------------
# EDA: 상관 최대 지연 108h (12개 horizon 전부 동일) -> 385 km/s
REFERENCE_TRANSIT_HOURS = 108.0
TRANSIT_SPEEDS = (315.0, 345.0, 385.0, 435.0, 500.0, 600.0)   # tau = 132/120/108/95/83/69 h
BALLISTIC_OFFSETS = (-4, -2, 0, 2)    # 소스 시각 기준 ±1일 (6h 스텝)
BALLISTIC_SOURCE = "window"   # "window" = 기준속도 ±offset / "speeds" = P3 방식(속도별 점 샘플)
BALLISTIC_LAT = "profile"     # "profile" = 중앙자오선 위도 전부 / "equator" = P3 방식(적도 1셀)

# ---- 모델 -----------------------------------------------------------------
USE_CH_GATHER = True         # 개선 1: 은닉 시퀀스 horizon 정렬 gather
USE_BALLISTIC_WINDOW = True  # 개선 2: 탄도 창 샘플 + 위도 프로파일
CH_HIDDEN = 64
CH_BIDIRECTIONAL = True      # 20프레임 전부 T0 에 관측됨 -> 인과 제약 없음
GATHER_DIM = 16
HORIZON_EMBED = 8
DROPOUT = 0.4
VERBOSE_MODEL = True         # CV 중에는 꺼서 출력을 줄인다

# ---- 학습 -----------------------------------------------------------------
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
GRAD_CLIP = 1.0
NUM_WORKERS = 4
LOSS_EPSILON = 1e-8
LOSS_SCALE = 100.0
CV_EPOCHS = 30               # CV 는 고정 epoch. early stopping 없음
N_FOLDS = 5
AUGMENT = True
AUG_CH_NOISE = 0.05

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

print("PyTorch:", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("data:", DATA_ROOT.resolve())

## 2. 데이터 로드 · 시간축 복원

`inputs.csv` 에 타임스탬프가 없고 **행 순서도 시간 순서가 아니다**(EDA 확인).
한 행의 `image_00..image_19` 가 연속 20시점이라는 사실만으로 프레임 시간축을 복원한다.
복원된 사슬이 곧 연속 관측 구간이고, CV 폴드의 단위가 된다.

In [ ]:
train_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")
train_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")
val_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")
val_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")
test_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")
assert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()
assert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()


def fill_wind(inputs):
    wind = inputs[WIND_COLUMNS].to_numpy(np.float64)
    valid = np.isfinite(wind).astype(np.float32)
    frame = pd.DataFrame(wind).ffill(axis=1).bfill(axis=1)
    filled = frame.to_numpy(np.float32)
    return np.nan_to_num(filled, nan=float(np.nanmedian(wind))), valid


train_wind, train_wind_valid = fill_wind(train_inputs)
val_wind, val_wind_valid = fill_wind(val_inputs)
test_wind, test_wind_valid = fill_wind(test_inputs)
train_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
val_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)


def reconstruct_frame_chains(inputs):
    """이미지 파일명만으로 프레임 시간축을 복원한다. 행 순서에 의존하지 않는다."""
    images = inputs[IMAGE_COLUMNS].to_numpy()
    successor, predecessor, conflicts = {}, {}, 0
    for row in images:
        for current, following in zip(row[:-1], row[1:]):
            if successor.setdefault(current, following) != following:
                conflicts += 1
            if predecessor.setdefault(following, current) != current:
                conflicts += 1
    names = set(images.ravel().tolist())
    chains, visited = [], set()
    for head in sorted(names - set(predecessor)):
        chain, node = [], head
        while node is not None and node not in visited:
            visited.add(node); chain.append(node); node = successor.get(node)
        chains.append(chain)
    assert conflicts == 0 and not (names - visited), "사슬 복원 실패"
    return chains


def sample_chain_index(inputs, chains):
    """샘플이 속한 사슬 번호와 사슬 내 시작 위치."""
    position = {name: (c, o)
                for c, chain in enumerate(chains) for o, name in enumerate(chain)}
    first = inputs[IMAGE_COLUMNS[0]].to_numpy()
    chain_id = np.array([position[n][0] for n in first])
    offset = np.array([position[n][1] for n in first])
    return chain_id, offset


TRAIN_CHAINS = reconstruct_frame_chains(train_inputs)
VAL_CHAINS = reconstruct_frame_chains(val_inputs)
TEST_CHAINS = reconstruct_frame_chains(test_inputs)
TRAIN_CHAIN_ID, TRAIN_OFFSET = sample_chain_index(train_inputs, TRAIN_CHAINS)

# 사슬 단위 파일 목록 (시간 순). 캐시 인덱스의 기준이 된다.
train_files = [n for chain in TRAIN_CHAINS for n in chain]
val_files = [n for chain in VAL_CHAINS for n in chain]
test_files = [n for chain in TEST_CHAINS for n in chain]
train_map = {n: i for i, n in enumerate(train_files)}
val_map = {n: i for i, n in enumerate(val_files)}
test_map = {n: i for i, n in enumerate(test_files)}

print(f"train {len(train_inputs):,} 샘플 / 사슬 {len(TRAIN_CHAINS)}개 / 고유 이미지 {len(train_files):,}")
print(f"val   {len(val_inputs):,} 샘플 / 사슬 {len(VAL_CHAINS)}개 / 고유 이미지 {len(val_files):,}")
print(f"test  {len(test_inputs):,} 샘플 / 사슬 {len(TEST_CHAINS)}개 / 고유 이미지 {len(test_files):,}")
print(f"\ntrain 사슬별 샘플 수: {np.bincount(TRAIN_CHAIN_ID).tolist()}")

## 3. 코로나홀 추출

**프레임마다 원반을 다시 검출한다.** EDA 에서 반지름의 프레임별 표준편차가 6.4% 로 나왔다.
고정 마스크를 쓰면 원반보다 마스크가 클 때 **배경(어두움)이 코로나홀로 오검출**된다.

원반 검출 임계는 최댓값 기준이 아니라 **분포 백분위 기준**으로 잡는다. 플레어가 최댓값을 끌어올려
마스크를 축소시키는 문제를 피하기 위해서다.

면적은 **원반 전체 픽셀 대비 비율**로 저장한다. 이렇게 하면 세밀 격자를 **합산**해 임의의 성긴 격자를
정확히 만들 수 있다 (셀별 픽셀 수가 달라도 오차가 없다).

In [ ]:
FINE_LAT, FINE_LON = FINE_GRID
FINE_CELLS = FINE_LAT * FINE_LON
N_LEVELS = len(CH_CUTS) + 1                     # 어두운 단계들 + 활성영역 1단계
LEVEL_NAMES = [f"dark{c}" for c in CH_CUTS] + ["bright"]


def load_pair(split, name):
    planes = []
    for channel in CHANNELS:
        with Image.open(DATA_ROOT / split / channel / name) as image:
            planes.append(np.asarray(image.convert("L"), dtype=np.float32))
    return np.stack(planes)


def detect_disk(frame):
    """플레어에 둔감한 원반 검출. 배경과 원반 내부의 중간값을 임계로 쓴다."""
    plane = frame.mean(axis=0)
    background = np.percentile(plane, 2.0)
    interior = np.percentile(plane, 70.0)
    mask = plane > background + 0.35 * (interior - background)
    ys, xs = np.nonzero(mask)
    return float(ys.mean()), float(xs.mean()), float(math.sqrt(mask.sum() / math.pi))


_geometry_cache = {}
GEOMETRY_CACHE_LIMIT = 96      # 항목당 약 0.5MB. 프레임마다 다시 만들면 느리므로 캐시한다.


def cell_geometry(side, center_y, center_x, radius):
    """(원반 마스크, 셀 번호) — 중심/반지름을 1px 로 양자화해 재사용한다."""
    key = (side, round(center_y), round(center_x), round(radius))
    entry = _geometry_cache.get(key)
    if entry is None:
        _, cy, cx, r = key
        grid_y, grid_x = np.mgrid[0:side, 0:side].astype(np.float32)
        disk = np.sqrt((grid_y - cy) ** 2 + (grid_x - cx) ** 2) <= r
        lat = np.clip((grid_y - (cy - r)) / (2 * r) * FINE_LAT, 0, FINE_LAT - 1e-4)
        lon = np.clip((grid_x - (cx - r)) / (2 * r) * FINE_LON, 0, FINE_LON - 1e-4)
        cell = (lat.astype(np.int32) * FINE_LON + lon.astype(np.int32))[disk]
        entry = (disk, cell)
        _geometry_cache[key] = entry
        while len(_geometry_cache) > GEOMETRY_CACHE_LIMIT:
            _geometry_cache.pop(next(iter(_geometry_cache)))
    return entry


def extract_split(split, filenames):
    """(n_files, N_LEVELS, FINE_CELLS) 원반 대비 면적 비율 + 프레임 기하 정보."""
    area = np.zeros((len(filenames), N_LEVELS, FINE_CELLS), np.float32)
    geometry = np.zeros((len(filenames), 4), np.float32)   # cy, cx, r, 원반픽셀수
    fixed = None
    if not PER_FRAME_DISK:
        sample = np.unique(np.linspace(0, len(filenames) - 1, 200).astype(int))
        measured = np.array([detect_disk(load_pair(split, filenames[i])) for i in sample])
        fixed = tuple(np.median(measured, axis=0))
        print(f"  고정 원반 사용: center=({fixed[0]:.1f},{fixed[1]:.1f}) r={fixed[2]:.1f}")
    started = time.perf_counter()
    for index, name in enumerate(filenames):
        frame = load_pair(split, name)
        center_y, center_x, radius = fixed if fixed is not None else detect_disk(frame)
        disk, cell = cell_geometry(frame.shape[-1], center_y, center_x, radius * DISK_MARGIN)
        on_disk = frame[:, disk]
        total = max(on_disk.shape[1], 1)
        geometry[index] = (center_y, center_x, radius, total)
        median = np.median(on_disk[:, ::4], axis=1, keepdims=True)
        normalized = on_disk / np.maximum(median, 1e-3)
        for level, cut in enumerate(CH_CUTS):
            selection = np.logical_and(normalized[0] <= cut, normalized[1] <= cut)
            area[index, level] = np.bincount(cell[selection], minlength=FINE_CELLS) / total
        selection = np.logical_and(normalized[0] >= BRIGHT_CUT, normalized[1] >= BRIGHT_CUT)
        area[index, -1] = np.bincount(cell[selection], minlength=FINE_CELLS) / total
        if (index + 1) % 2000 == 0 or index + 1 == len(filenames):
            print(f"  {split} {index + 1}/{len(filenames)} "
                  f"({time.perf_counter() - started:.0f}s)", flush=True)
    return area, geometry


def cached_extract(split, filenames):
    tag = (f"{CH_CODE_VERSION}_{FINE_LAT}x{FINE_LON}"
           f"_c{'-'.join(str(c) for c in CH_CUTS)}_b{BRIGHT_CUT}_m{DISK_MARGIN}"
           f"_{'perframe' if PER_FRAME_DISK else 'fixed'}")
    area_path = CACHE_ROOT / f"ch_{split}_{tag}.npy"
    geometry_path = CACHE_ROOT / f"geom_{split}_{tag}.npy"
    if area_path.exists() and geometry_path.exists():
        area = np.load(area_path)
        if area.shape == (len(filenames), N_LEVELS, FINE_CELLS):
            print(f"캐시 재사용: {area_path.name}")
            return area, np.load(geometry_path)
    area, geometry = extract_split(split, filenames)
    np.save(area_path, area); np.save(geometry_path, geometry)
    return area, geometry


train_area, train_geometry = cached_extract("train", train_files)
val_area, val_geometry = cached_extract("validation", val_files)
test_area, test_geometry = cached_extract("test", test_files)

for name, geometry in [("train", train_geometry), ("val", val_geometry), ("test", test_geometry)]:
    radius = geometry[:, 2]
    print(f"{name:5s} 반지름 {radius.mean():.1f} ± {radius.std():.1f}px "
          f"({radius.std()/radius.mean():.2%}), 원반 픽셀 {geometry[:, 3].mean():,.0f}")
print("\n원반 대비 면적 비율 (train 평균):")
for level, label in enumerate(LEVEL_NAMES):
    print(f"  {label:10s} {train_area[:, level].sum(axis=1).mean():.4f}")

## 4. 격자 집계 · 탄도 정렬

세밀 격자를 `CH_GRID` 로 합산한다. 면적이 원반 대비 비율이므로 **단순 합이 정확하다.**

`FOLD_LATITUDE` 가 켜져 있으면 적도 대칭 밴드를 합친다 (Collin 2025 의 $S_{i,j}=A_{i,j}+A_{m-i+1,j}$).
남북 비대칭의 올바른 사용은 지구 태양위도 $\alpha$ 를 알아야 하는데 그건 외부 데이터라 쓸 수 없다.
대칭화하면 학습 불가능한 성분을 제거하는 효과가 있다.

**탄도 인덱스는 EDA 로 재설정했다.** 경험적 최적 지연 108h 를 중심으로 속도를 배치한다.
P3 의 v=700 은 60h 이상 horizon 에서 인덱스가 19 로 clamp 되어 죽어 있었다.

In [ ]:
def aggregate(area, grid_lat, grid_lon, fold):
    """(n, L, FINE_CELLS) -> (n, L, cells). 면적이 원반 대비 비율이라 단순 합이 정확하다."""
    block = area.reshape(len(area), N_LEVELS, FINE_LAT, FINE_LON)
    block = block.reshape(len(area), N_LEVELS, grid_lat, FINE_LAT // grid_lat,
                          grid_lon, FINE_LON // grid_lon).sum(axis=(3, 5))
    if fold:
        block = block + block[:, :, ::-1, :]           # 적도 대칭 합 (Collin 의 S_ij)
        block = block[:, :, : (grid_lat + 1) // 2, :]
    return np.ascontiguousarray(block.reshape(len(area), N_LEVELS, -1), dtype=np.float32)


def ballistic_indices(speeds):
    table = np.zeros((12, len(speeds)), np.float32)
    for h in range(12):
        lead = (h + 1) * 6.0
        for s, speed in enumerate(speeds):
            table[h, s] = np.clip(19.0 + (lead - AU_KM / speed / 3600.0) / 6.0, 0.0, 19.0)
    return table


def image_index_matrix(inputs, image_map):
    return np.asarray([[image_map[n] for n in row]
                       for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)],
                      dtype=np.int64)


def interpolate_at(sequence, indices):
    """sequence (n, 20, D) 를 실수 인덱스에서 선형보간한다."""
    flat = np.asarray(indices, np.float32).ravel()
    lower = np.floor(flat).astype(np.int64)
    upper = np.minimum(lower + 1, 19)
    weight = (flat - lower).astype(np.float32)[None, :, None]
    picked = sequence[:, lower] * (1.0 - weight) + sequence[:, upper] * weight
    return picked.reshape(len(sequence), *np.shape(indices), sequence.shape[2])


def flatten_ch(grid, indexes):
    return grid[indexes].reshape(len(indexes), 20, N_USED_LEVELS * N_CELLS).astype(np.float32)


def ballistic_area(grid, indexes):
    """탄도 소스 시각에서 뽑은 코로나홀 면적 -> (n, 12, BALLISTIC_DIM)."""
    columns = [lat * GRID_LON + CENTRAL_LON for lat in AREA_LAT_ROWS]
    profile = grid[indexes][:, :, :, columns]
    profile = profile.reshape(len(indexes), 20, N_USED_LEVELS * len(AREA_LAT_ROWS))
    picked = interpolate_at(profile, AREA_INDEX)
    return picked.reshape(len(indexes), 12, BALLISTIC_DIM).astype(np.float32)


train_index_matrix = image_index_matrix(train_inputs, train_map)
val_index_matrix = image_index_matrix(val_inputs, val_map)
test_index_matrix = image_index_matrix(test_inputs, test_map)


def configure(**overrides):
    """설정을 바꾸고 파생 피처를 다시 만든다. ablation 은 이 함수로만 한다."""
    globals().update(overrides)
    global GRID_LAT, GRID_LON, USED_LAT, N_CELLS, CENTRAL_LON, EQUATOR_ROW
    global LEVEL_INDEX, N_USED_LEVELS, CH_SEQ_DIM
    global train_grid, val_grid, test_grid
    global BALLISTIC_INDEX, N_SPEEDS, REFERENCE_INDEX, WINDOW_INDEX
    global AREA_INDEX, AREA_LAT_ROWS, BALLISTIC_DIM

    GRID_LAT, GRID_LON = CH_GRID
    assert FINE_LAT % GRID_LAT == 0 and FINE_LON % GRID_LON == 0
    USED_LAT = (GRID_LAT + 1) // 2 if FOLD_LATITUDE else GRID_LAT
    N_CELLS = USED_LAT * GRID_LON
    CENTRAL_LON = GRID_LON // 2
    EQUATOR_ROW = USED_LAT - 1 if FOLD_LATITUDE else GRID_LAT // 2
    LEVEL_INDEX = [LEVEL_NAMES.index(name) for name in USE_LEVELS]
    N_USED_LEVELS = len(LEVEL_INDEX)
    CH_SEQ_DIM = N_USED_LEVELS * N_CELLS

    train_grid = aggregate(train_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]
    val_grid = aggregate(val_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]
    test_grid = aggregate(test_area, GRID_LAT, GRID_LON, FOLD_LATITUDE)[:, LEVEL_INDEX]

    BALLISTIC_INDEX = ballistic_indices(TRANSIT_SPEEDS)
    N_SPEEDS = len(TRANSIT_SPEEDS)
    REFERENCE_INDEX = np.clip(19.0 + (HORIZONS - REFERENCE_TRANSIT_HOURS) / 6.0, 0.0, 19.0)
    WINDOW_INDEX = np.clip(REFERENCE_INDEX[:, None] + np.asarray(BALLISTIC_OFFSETS)[None, :],
                           0.0, 19.0).astype(np.float32)
    AREA_INDEX = WINDOW_INDEX if BALLISTIC_SOURCE == "window" else BALLISTIC_INDEX
    AREA_LAT_ROWS = list(range(USED_LAT)) if BALLISTIC_LAT == "profile" else [EQUATOR_ROW]
    BALLISTIC_DIM = AREA_INDEX.shape[1] * len(AREA_LAT_ROWS) * N_USED_LEVELS


configure()
print(f"격자 {GRID_LAT}x{GRID_LON} -> 셀 {N_CELLS} (적도 대칭 접기 {FOLD_LATITUDE}), 레벨 {USE_LEVELS}")
print(f"ch_seq 차원 {CH_SEQ_DIM}, 탄도 면적 차원 {BALLISTIC_DIM}, gather 속도 {N_SPEEDS}개")
print("\n탄도 소스 인덱스 (19 = 마지막 관측). EDA 경험적 최적은 horizon 별 2 -> 13 이다.")
print(pd.DataFrame(BALLISTIC_INDEX, index=[f"{h}h" for h in HORIZONS],
                   columns=[f"v={int(v)}" for v in TRANSIT_SPEEDS]).round(2))
print(f"\n면적 샘플 인덱스 (source={BALLISTIC_SOURCE}, lat={BALLISTIC_LAT})")
print(pd.DataFrame(np.asarray(AREA_INDEX), index=[f"{h}h" for h in HORIZONS]).round(2))

## 5. 정규화 통계 · Dataset

CV 폴드마다 **그 폴드의 학습 행에서만** 통계를 다시 낸다. 최종 학습에서는 train 전체를 쓴다.

In [ ]:
STAT_NAMES = ["last", "mean4", "mean", "std", "min", "max", "slope", "last_minus_mean4", "range"]
NUM_STATS = len(STAT_NAMES)
_TIME_CENTERED = np.arange(20, dtype=np.float32) - 9.5
_TIME_DENOM = float((_TIME_CENTERED ** 2).sum())


def build_wind_stats(wind):
    last = wind[:, -1]
    mean4 = wind[:, -4:].mean(axis=1)
    slope = (wind - wind.mean(axis=1, keepdims=True)) @ _TIME_CENTERED / _TIME_DENOM
    return np.stack([last, mean4, wind.mean(axis=1), wind.std(axis=1), wind.min(axis=1),
                     wind.max(axis=1), slope, last - mean4,
                     wind.max(axis=1) - wind.min(axis=1)], axis=1).astype(np.float32)


def fit_stats(rows):
    """주어진 train 행에서만 정규화/복원 통계를 산출한다."""
    wind = train_wind[rows]
    targets = train_targets[rows]
    ch_seq = flatten_ch(train_grid, train_index_matrix[rows]).reshape(-1, CH_SEQ_DIM)
    ballistic = ballistic_area(train_grid, train_index_matrix[rows]).reshape(-1, BALLISTIC_DIM)
    statistics = build_wind_stats(wind)
    residual = targets - wind[:, -1:]
    return {
        "wind_mean": float(wind.mean()), "wind_std": float(wind.std() + 1e-6),
        "diff_std": float(np.diff(wind, axis=1, prepend=wind[:, :1]).std() + 1e-6),
        "stats_mean": statistics.mean(axis=0), "stats_std": statistics.std(axis=0) + 1e-6,
        "ch_mean": ch_seq.mean(axis=0), "ch_std": ch_seq.std(axis=0) + 1e-8,
        "ballistic_mean": ballistic.mean(axis=0), "ballistic_std": ballistic.std(axis=0) + 1e-8,
        "residual_mean": residual.mean(axis=0), "residual_std": residual.std(axis=0) + 1e-6,
        "clip_low": float(targets.min() * 0.95), "clip_high": float(targets.max() * 1.05),
    }


class SolarWindDataset(Dataset):
    def __init__(self, inputs, index_matrix, wind, wind_valid, grid, stats,
                 targets=None, training=False):
        self.training = training
        self.stats = stats
        self.sample_ids = inputs.sample_id.to_numpy()
        self.last_wind = np.ascontiguousarray(wind[:, -1]).astype(np.float32)
        self.wind_seq = np.stack([
            (wind - stats["wind_mean"]) / stats["wind_std"],
            np.diff(wind, axis=1, prepend=wind[:, :1]) / stats["diff_std"],
            wind_valid], axis=2).astype(np.float32)
        self.wind_stats = ((build_wind_stats(wind) - stats["stats_mean"])
                           / stats["stats_std"]).astype(np.float32)
        self.ch_seq = ((flatten_ch(grid, index_matrix) - stats["ch_mean"])
                       / stats["ch_std"]).astype(np.float32)
        self.ballistic = ((ballistic_area(grid, index_matrix) - stats["ballistic_mean"])
                          / stats["ballistic_std"]).astype(np.float32)
        self.targets = targets.astype(np.float32) if targets is not None else None

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, item):
        ch_seq, ballistic = self.ch_seq[item], self.ballistic[item]
        if self.training and AUGMENT and AUG_CH_NOISE > 0:
            ch_seq = ch_seq * (1.0 + np.random.normal(0, AUG_CH_NOISE, ch_seq.shape)
                               ).astype(np.float32)
            ballistic = ballistic * (1.0 + np.random.normal(0, AUG_CH_NOISE, ballistic.shape)
                                     ).astype(np.float32)
        result = {"wind_seq": torch.from_numpy(self.wind_seq[item]),
                  "wind_stats": torch.from_numpy(self.wind_stats[item]),
                  "ch_seq": torch.from_numpy(np.ascontiguousarray(ch_seq)),
                  "ballistic": torch.from_numpy(np.ascontiguousarray(ballistic)),
                  "last_wind": torch.tensor(self.last_wind[item]),
                  "sample_id": self.sample_ids[item]}
        if self.targets is not None:
            result["target"] = torch.from_numpy(self.targets[item])
        return result


def make_loader(dataset, shuffle, seed=SEED):
    options = dict(dataset=dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                   num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False,
                   generator=torch.Generator().manual_seed(seed))
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


_probe_stats = fit_stats(np.arange(len(train_inputs)))
print({k: (np.round(v, 3).tolist() if isinstance(v, np.ndarray) and v.size <= 6 else
           (v if np.isscalar(v) else f"array{np.shape(v)}"))
       for k, v in _probe_stats.items()})

## 6. 모델

P3 대비 바뀐 곳은 **CH 브랜치**뿐이다.

- CH GRU 를 **양방향**으로 바꾸고 **은닉 시퀀스 전체**를 받는다.
  20 프레임은 전부 T0 에 관측된 것이라 인과 제약이 없다. 역방향 패스는 소스 시각 *이후*의
  코로나홀 성장/소멸까지 요약에 넣는다.
- horizon 마다 탄도 인덱스에서 은닉 시퀀스를 **선형보간해 뽑는다**(gather).
  P3 에서 head 의 horizon 별 입력은 스칼라 3개뿐이었다.
- 탄도 창 샘플(중앙자오선 위도 프로파일 × 4시점)을 함께 넣는다.

head 는 12 horizon 에 **가중치를 공유**한다. horizon 을 구분하는 것은 embedding 과 위 두 입력이다.

In [ ]:
class SolarWindP7(nn.Module):
    def __init__(self, stats):
        super().__init__()
        self.wind_gru = nn.GRU(3, 96, num_layers=2, batch_first=True)
        self.stats_encoder = nn.Sequential(
            nn.Linear(NUM_STATS, 128), nn.SELU(inplace=True),
            nn.Linear(128, 64), nn.SELU(inplace=True))
        shared_dim = 96 + 64

        self.ch_gru = nn.GRU(CH_SEQ_DIM, CH_HIDDEN, num_layers=2, batch_first=True,
                             bidirectional=CH_BIDIRECTIONAL)
        directions = 2 if CH_BIDIRECTIONAL else 1
        self.ch_dropout = nn.Dropout(DROPOUT)
        shared_dim += CH_HIDDEN * directions

        head_extra = 0
        if USE_CH_GATHER:
            self.gather_project = nn.Sequential(
                nn.Linear(CH_HIDDEN * directions, GATHER_DIM), nn.ReLU(inplace=True))
            head_extra += GATHER_DIM * N_SPEEDS
        if USE_BALLISTIC_WINDOW:
            head_extra += BALLISTIC_DIM

        self.horizon_embedding = nn.Parameter(torch.randn(12, HORIZON_EMBED) * 0.1)
        head_input = shared_dim + HORIZON_EMBED + head_extra
        self.head = nn.Sequential(
            nn.Linear(head_input, 192), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(192, 96), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(96, 1))

        lower = np.floor(BALLISTIC_INDEX).astype(np.int64)
        self.register_buffer("gather_lower", torch.as_tensor(lower))
        self.register_buffer("gather_upper", torch.as_tensor(np.minimum(lower + 1, 19)))
        self.register_buffer("gather_weight",
                             torch.as_tensor(BALLISTIC_INDEX - lower, dtype=torch.float32))
        self.register_buffer("residual_mean", torch.as_tensor(stats["residual_mean"]))
        self.register_buffer("residual_std", torch.as_tensor(stats["residual_std"]))
        if VERBOSE_MODEL:
            print(f"shared={shared_dim} head_input={head_input} "
                  f"(gather {GATHER_DIM * N_SPEEDS if USE_CH_GATHER else 0}, "
                  f"ballistic {BALLISTIC_DIM if USE_BALLISTIC_WINDOW else 0})")

    def forward(self, wind_seq, wind_stats, ch_seq, ballistic):
        _, wind_hidden = self.wind_gru(wind_seq)
        ch_sequence, ch_hidden = self.ch_gru(ch_seq)
        if CH_BIDIRECTIONAL:
            ch_last = torch.cat([ch_hidden[-2], ch_hidden[-1]], dim=1)
        else:
            ch_last = ch_hidden[-1]
        parts = [F.relu(wind_hidden[-1]), self.stats_encoder(wind_stats),
                 self.ch_dropout(F.relu(ch_last))]

        shared = torch.cat(parts, dim=1)
        batch = shared.shape[0]
        head_parts = [shared.unsqueeze(1).expand(batch, 12, shared.shape[1]),
                      self.horizon_embedding.unsqueeze(0).expand(batch, 12, HORIZON_EMBED)]
        if USE_CH_GATHER:
            projected = self.gather_project(ch_sequence)               # (B, 20, G)
            weight = self.gather_weight.unsqueeze(-1)                  # (12, S, 1)
            gathered = (projected[:, self.gather_lower] * (1.0 - weight)
                        + projected[:, self.gather_upper] * weight)    # (B, 12, S, G)
            head_parts.append(gathered.flatten(2))
        if USE_BALLISTIC_WINDOW:
            head_parts.append(ballistic)
        z = self.head(torch.cat(head_parts, dim=2)).squeeze(-1)
        return z * self.residual_std + self.residual_mean


def build_model(stats):
    return SolarWindP7(stats).to(DEVICE)


_probe = build_model(_probe_stats)
print("trainable parameters:",
      f"{sum(p.numel() for p in _probe.parameters() if p.requires_grad):,}")
del _probe; gc.collect()

## 7. 지표와 손실

평가지표는 horizon 별 RMSE 를 낸 뒤 평균한 값이다. 전체 원소를 묶은 pooled RMSE 와 다르므로
손실도 같은 형태로 맞춘다.

In [ ]:
def official_rmse(y_true, y_pred):
    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    return float(per_horizon.mean()), per_horizon


def metric_loss(prediction, target):
    error = (prediction - target) / LOSS_SCALE
    return torch.sqrt((error ** 2).mean(dim=0) + LOSS_EPSILON).mean()


@torch.no_grad()
def predict_with(model, loader, clip_low, clip_high):
    model.eval()
    predictions, sample_ids = [], []
    for batch in loader:
        moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)
                 for k in ("wind_seq", "wind_stats", "ch_seq", "ballistic", "last_wind")}
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            residual = model(moved["wind_seq"], moved["wind_stats"],
                             moved["ch_seq"], moved["ballistic"])
        prediction = (residual.float() + moved["last_wind"].unsqueeze(1)
                      ).clamp(clip_low, clip_high)
        predictions.append(prediction.cpu().numpy())
        sample_ids.extend(batch["sample_id"])
    return np.concatenate(predictions).astype(np.float64), sample_ids


PERSISTENCE_VAL = official_rmse(val_targets, np.repeat(val_wind[:, -1:], 12, axis=1))
print(f"validation persistence 기준선: {PERSISTENCE_VAL[0]:.3f} km/s")

## 8. Train 시간블록 CV — 계측기

validation 은 **9월만 11개** 모인 구간이고 test 는 10~12월이다. 계절이 다르고, 이미 여러 세대에 걸쳐
모델 선택에 소진됐다. 판단은 train 내부 CV 로 한다.

- 폴드 단위는 **복원된 사슬**(연속 관측 구간)이다. 사슬 내부는 연속이고 사슬 사이에는 실제 관측 공백이 있다
- 사슬을 통째로 배정하므로 윈도우 중첩에 의한 누수가 없다
- **early stopping 을 쓰지 않는다.** 고정 epoch 로 돌리고 매 epoch 의 홀드아웃 점수를 기록만 한다.
  epoch 선택은 전 폴드 평균 곡선에서 **한 번만** 한다
- **horizon 별로 보고한다.** 전체 평균만 보면 장기 개선이 단기 악화에 가려진다

In [ ]:
def build_folds(n_folds=N_FOLDS):
    """사슬을 샘플 수가 고르게 되도록 폴드에 배정한다."""
    counts = np.bincount(TRAIN_CHAIN_ID, minlength=len(TRAIN_CHAINS))
    order = np.argsort(counts)[::-1]
    assignment = np.zeros(len(TRAIN_CHAINS), np.int64)
    loads = np.zeros(n_folds, np.int64)
    for chain in order:
        fold = int(np.argmin(loads))
        assignment[chain] = fold
        loads[fold] += counts[chain]
    folds = []
    for fold in range(n_folds):
        evaluate = np.flatnonzero(assignment[TRAIN_CHAIN_ID] == fold)
        train = np.flatnonzero(assignment[TRAIN_CHAIN_ID] != fold)
        folds.append((train, evaluate))
    return folds


FOLDS = build_folds()
print("폴드별 (학습, 평가) 샘플 수:", [(len(a), len(b)) for a, b in FOLDS])


def train_run(train_rows, evaluate_rows, seed=SEED, epochs=CV_EPOCHS, verbose=False):
    """고정 epoch 학습. 매 epoch 의 홀드아웃 horizon별 RMSE 를 기록만 한다."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    stats = fit_stats(train_rows)
    train_dataset = SolarWindDataset(
        train_inputs.iloc[train_rows], train_index_matrix[train_rows], train_wind[train_rows],
        train_wind_valid[train_rows], train_grid, stats, train_targets[train_rows], training=True)
    evaluate_dataset = SolarWindDataset(
        train_inputs.iloc[evaluate_rows], train_index_matrix[evaluate_rows],
        train_wind[evaluate_rows], train_wind_valid[evaluate_rows], train_grid, stats,
        train_targets[evaluate_rows])
    train_loader = make_loader(train_dataset, True, seed)
    evaluate_loader = make_loader(evaluate_dataset, False, seed)
    evaluate_targets = train_targets[evaluate_rows]

    model = build_model(stats)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    curve = np.zeros((epochs, 12), np.float64)
    states = []
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)
                     for k in ("wind_seq", "wind_stats", "ch_seq", "ballistic",
                               "last_wind", "target")}
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                residual = model(moved["wind_seq"], moved["wind_stats"],
                                 moved["ch_seq"], moved["ballistic"])
            loss = metric_loss(residual.float() + moved["last_wind"].unsqueeze(1),
                               moved["target"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
        scheduler.step()
        prediction, _ = predict_with(model, evaluate_loader,
                                     stats["clip_low"], stats["clip_high"])
        _, per_horizon = official_rmse(evaluate_targets, prediction)
        curve[epoch] = per_horizon
        if verbose:
            print(f"    epoch {epoch + 1:03d} rmse {per_horizon.mean():7.3f} "
                  f"(72h {per_horizon[-1]:6.2f})", flush=True)
    del model, train_loader, evaluate_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return curve


def run_cv(label, folds=FOLDS, seeds=(SEED,), epochs=CV_EPOCHS):
    started = time.perf_counter()
    curves = np.zeros((len(folds), len(seeds), epochs, 12))
    for f, (train_rows, evaluate_rows) in enumerate(folds):
        for s, seed in enumerate(seeds):
            curves[f, s] = train_run(train_rows, evaluate_rows, seed, epochs)
            print(f"  fold {f + 1}/{len(folds)} seed {seed}: "
                  f"best {curves[f, s].mean(axis=1).min():.3f} "
                  f"({time.perf_counter() - started:.0f}s)", flush=True)
    mean_curve = curves.mean(axis=(0, 1))                    # (epochs, 12)
    best_epoch = int(np.argmin(mean_curve.mean(axis=1)))
    np.save(OUTPUT_DIR / f"cv_{label}.npy", curves)
    print(f"\n[{label}] best epoch {best_epoch + 1} / CV RMSE "
          f"{mean_curve[best_epoch].mean():.3f}  ({time.perf_counter() - started:.0f}s)")
    print(pd.DataFrame({"horizon": HORIZONS, "cv_rmse": mean_curve[best_epoch].round(2),
                        "fold_std": curves[:, :, best_epoch].reshape(-1, 12).std(axis=0).round(2)}
                       ).to_string(index=False))
    return curves, best_epoch, mean_curve


# 누적 ablation. A 가 P3 재현이고, 그 위에 하나씩만 얹는다.
# 시간이 부족하면 뒤에서부터 잘라도 된다 (앞 단계가 더 근거가 확실하다).
ABLATION_LADDER = [
    ("A. P3 재현", dict(CH_GRID=(3, 5), FOLD_LATITUDE=False, USE_LEVELS=("dark0.45",),
                        TRANSIT_SPEEDS=(350.0, 500.0, 700.0),
                        BALLISTIC_SOURCE="speeds", BALLISTIC_LAT="equator",
                        BALLISTIC_OFFSETS=(0,), USE_CH_GATHER=False,
                        USE_BALLISTIC_WINDOW=True, CH_BIDIRECTIONAL=False)),
    ("B. + 탄도속도 재설정", dict(TRANSIT_SPEEDS=(315.0, 385.0, 500.0))),
    ("C. + 격자축·적도대칭", dict(CH_GRID=(6, 3), FOLD_LATITUDE=True)),
    ("D. + 활성영역 레벨", dict(USE_LEVELS=("dark0.45", "bright"))),
    ("E. + 탄도창·위도프로파일", dict(BALLISTIC_SOURCE="window", BALLISTIC_LAT="profile",
                                     BALLISTIC_OFFSETS=(-4, -2, 0, 2))),
    ("F. + horizon 정렬 gather", dict(USE_CH_GATHER=True, CH_BIDIRECTIONAL=True,
                                      TRANSIT_SPEEDS=(315.0, 345.0, 385.0,
                                                      435.0, 500.0, 600.0))),
]

VERBOSE_MODEL = False
RESULTS, settings = [], {}
for label, overrides in ABLATION_LADDER:
    settings.update(overrides)
    configure(**settings)
    print()
    print(f"=== {label} ===  ch_seq {CH_SEQ_DIM} / 탄도 {BALLISTIC_DIM} / "
          f"속도 {N_SPEEDS}", flush=True)
    curves, best_epoch, mean_curve = run_cv(label.split(".")[0], epochs=CV_EPOCHS)
    RESULTS.append({"label": label, "settings": dict(settings), "curves": curves,
                    "best_epoch": best_epoch, "mean": mean_curve})
VERBOSE_MODEL = True

rows = []
for result in RESULTS:
    per_horizon = result["mean"][result["best_epoch"]]
    fold_scores = result["curves"][:, :, result["best_epoch"]].reshape(-1, 12).mean(axis=1)
    rows.append({"config": result["label"], "epoch": result["best_epoch"] + 1,
                 "CV": per_horizon.mean(), "6-36h": per_horizon[:6].mean(),
                 "42-72h": per_horizon[6:].mean(), "72h": per_horizon[-1],
                 "fold_se": fold_scores.std(ddof=1) / np.sqrt(len(fold_scores))})
summary = pd.DataFrame(rows)
summary["dCV"] = summary.CV.diff()
summary["d42-72h"] = summary["42-72h"].diff()
print()
print("=" * 78)
print(summary.round(3).to_string(index=False))
print("=" * 78)
print("dCV / d42-72h 가 음수면 그 단계가 개선이다.")
print("fold_se 의 2배보다 작은 변화는 노이즈로 보고 더 단순한 쪽(위 단계)을 택할 것.")
summary.to_csv(OUTPUT_DIR / "ablation.csv", index=False)

BEST_RESULT = min(RESULTS, key=lambda r: r["mean"][r["best_epoch"]].mean())
configure(**BEST_RESULT["settings"])
CV_CURVES = BEST_RESULT["curves"]
BEST_EPOCH = BEST_RESULT["best_epoch"]
CV_MEAN = BEST_RESULT["mean"]
print()
print(f"채택: {BEST_RESULT['label']}  (CV {CV_MEAN[BEST_EPOCH].mean():.3f}, "
      f"epoch {BEST_EPOCH + 1})")
print(json.dumps({k: str(v) for k, v in BEST_RESULT["settings"].items()},
                 ensure_ascii=False, indent=2))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for result in RESULTS:
    axes[0].plot(np.arange(1, CV_EPOCHS + 1), result["mean"].mean(axis=1),
                 label=result["label"].split(".")[0])
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("CV RMSE [km/s]")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3); axes[0].set_title("CV learning curve")
for result in (RESULTS[0], BEST_RESULT):
    axes[1].plot(HORIZONS, result["mean"][result["best_epoch"]], "o-",
                 label=result["label"].split(".")[0])
axes[1].set_xlabel("horizon [hour]"); axes[1].set_ylabel("RMSE [km/s]")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3); axes[1].set_title("CV RMSE by horizon")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "cv.png", dpi=140); plt.show()

## 9. 최종 학습

CV 가 정한 epoch 수로 **train 전체**에 대해 한 번 학습한다.
validation 은 학습에 쓰지 않는다 (규정).

In [ ]:
FINAL_EPOCHS = BEST_EPOCH + 1
print(f"최종 학습: train 전체 {len(train_inputs):,} 샘플, {FINAL_EPOCHS} epoch")

FINAL_STATS = fit_stats(np.arange(len(train_inputs)))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

final_train = SolarWindDataset(train_inputs, train_index_matrix, train_wind, train_wind_valid,
                               train_grid, FINAL_STATS, train_targets, training=True)
final_loader = make_loader(final_train, True, SEED)

model = build_model(FINAL_STATS)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# CV 와 동일한 LR 궤적을 쓰기 위해 T_max 는 CV_EPOCHS 로 둔다 (FINAL_EPOCHS 에서 멈출 뿐이다)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CV_EPOCHS)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

history = []
for epoch in range(FINAL_EPOCHS):
    started = time.perf_counter()
    model.train()
    squared_error, count = np.zeros(12), 0
    for batch in final_loader:
        moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)
                 for k in ("wind_seq", "wind_stats", "ch_seq", "ballistic",
                           "last_wind", "target")}
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            residual = model(moved["wind_seq"], moved["wind_stats"],
                             moved["ch_seq"], moved["ballistic"])
        prediction = residual.float() + moved["last_wind"].unsqueeze(1)
        loss = metric_loss(prediction, moved["target"])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        error = (prediction.detach() - moved["target"]).double()
        squared_error += torch.sum(error ** 2, dim=0).cpu().numpy()
        count += error.shape[0]
    scheduler.step()
    train_rmse = float(np.sqrt(squared_error / count).mean())
    history.append({"epoch": epoch + 1, "train_rmse": train_rmse,
                    "seconds": time.perf_counter() - started})
    print(f"epoch {epoch + 1:03d} train {train_rmse:7.3f} "
          f"lr {optimizer.param_groups[0]['lr']:.2e} "
          f"{history[-1]['seconds']:5.1f}s", flush=True)

pd.DataFrame(history).to_csv(OUTPUT_DIR / "history.csv", index=False)

## 10. Validation 확인 (판단용 아님) · Test 추론

In [ ]:
val_dataset = SolarWindDataset(val_inputs, val_index_matrix, val_wind, val_wind_valid,
                               val_grid, FINAL_STATS, val_targets)
val_prediction, val_ids = predict_with(model, make_loader(val_dataset, False),
                                       FINAL_STATS["clip_low"], FINAL_STATS["clip_high"])
assert val_ids == val_inputs.sample_id.tolist()
val_score, val_per_horizon = official_rmse(val_targets, val_prediction)

P3_PER_HORIZON = np.array([28.35, 44.22, 53.62, 59.68, 64.32, 68.00,
                           70.64, 72.72, 74.44, 76.22, 78.20, 80.02])
comparison = pd.DataFrame({
    "horizon": HORIZONS, "P3": P3_PER_HORIZON, "P7": val_per_horizon.round(2),
    "delta": (val_per_horizon - P3_PER_HORIZON).round(2),
    "P7_cv": CV_MEAN[BEST_EPOCH].round(2),
})
print(f"validation RMSE  P3 {P3_PER_HORIZON.mean():.3f} -> P7 {val_score:.3f} "
      f"(persistence {PERSISTENCE_VAL[0]:.3f})")
print(comparison.to_string(index=False))
print("\n주의: validation 은 9월만 모인 구간이고 test 는 10~12월이다.")
print("      이 표는 확인용이며, 채택 판단은 CV 로 한다.")
comparison.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)

test_dataset = SolarWindDataset(test_inputs, test_index_matrix, test_wind, test_wind_valid,
                                test_grid, FINAL_STATS)
test_prediction, test_ids = predict_with(model, make_loader(test_dataset, False),
                                         FINAL_STATS["clip_low"], FINAL_STATS["clip_high"])
assert test_ids == test_inputs.sample_id.tolist()
assert test_prediction.shape == (len(test_inputs), 12) and np.isfinite(test_prediction).all()

submission = pd.DataFrame(test_prediction, columns=TARGET_COLUMNS)
submission.insert(0, "sample_id", test_inputs.sample_id.tolist())
submission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

CONFIG = {"ch_code_version": CH_CODE_VERSION, "fine_grid": list(FINE_GRID),
          "ch_grid": list(CH_GRID), "fold_latitude": FOLD_LATITUDE,
          "ch_cuts": list(CH_CUTS), "bright_cut": BRIGHT_CUT, "disk_margin": DISK_MARGIN,
          "per_frame_disk": PER_FRAME_DISK, "used_levels": list(USE_LEVELS),
          "transit_speeds": list(TRANSIT_SPEEDS),
          "reference_transit_hours": REFERENCE_TRANSIT_HOURS,
          "ballistic_offsets": list(BALLISTIC_OFFSETS),
          "use_ch_gather": USE_CH_GATHER, "use_ballistic_window": USE_BALLISTIC_WINDOW,
          "ch_bidirectional": CH_BIDIRECTIONAL, "gather_dim": GATHER_DIM,
          "seed": SEED, "epochs": FINAL_EPOCHS, "cv_rmse": float(CV_MEAN[BEST_EPOCH].mean()),
          "val_rmse": val_score, "initialization": "random_from_scratch",
          **{k: (v.tolist() if isinstance(v, np.ndarray) else v)
             for k, v in FINAL_STATS.items()}}
torch.save({"model_state_dict": model.state_dict(), **CONFIG},
           SUBMISSION_DIR / "model.pth")
print(f"\nsubmission.csv {submission.shape}, model.pth 저장 완료")
print(submission[TARGET_COLUMNS].describe().loc[["mean", "std", "min", "max"]].round(1))

## 11. 제출 점검

`submission/` 아래 `code.ipynb`, `model.pth`, `submission.csv` 세 파일이 필요하다.

> ⚠️ `code.ipynb` 는 **수동 복사**다. 이 노트북을 저장(Ctrl+S)한 뒤 `submission/code.ipynb` 로 복사할 것.

In [ ]:
EXPECTED_TEST_ROWS = 3868
ok = True
for name in ["code.ipynb", "model.pth", "submission.csv"]:
    path = SUBMISSION_DIR / name
    if path.exists():
        print(f"  {name:16s} {path.stat().st_size / 1024 ** 2:8.2f} MiB")
    else:
        print(f"  {name:16s} 없음"); ok = False

check = pd.read_csv(SUBMISSION_DIR / "submission.csv")
ok &= len(check) == EXPECTED_TEST_ROWS
ok &= list(check.columns) == ["sample_id"] + TARGET_COLUMNS
ok &= check.sample_id.tolist() == test_inputs.sample_id.tolist()
ok &= int(check.isna().sum().sum()) == 0
values = check[TARGET_COLUMNS].to_numpy()
ok &= bool((values >= FINAL_STATS["clip_low"] - 1e-6).all()
           and (values <= FINAL_STATS["clip_high"] + 1e-6).all())
print(f"\n행 수 {len(check)} / 결측 {int(check.isna().sum().sum())} / "
      f"값 범위 {values.min():.1f} ~ {values.max():.1f}")
print("최종:", "통과" if ok else "실패 — 위 항목 확인")